<a href="https://colab.research.google.com/github/E-tech-coder/heart-disease-prediction/blob/main/Predicting_Heart_Disease.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

playground_series_s6e2_path = kagglehub.competition_download('playground-series-s6e2')
print('Data source import complete.')
print(playground_series_s6e2_path)

# 1. Problem Definition

Cardiovascular disease is one of the leading causes of death worldwide. Early prediction of heart disease can help healthcare professionals identify high-risk patients and intervene earlier.

In this project, the goal is to build a machine learning model that predicts whether a patient is likely to develop heart disease based on clinical and demographic features.

Kaggle link : https://www.kaggle.com/competitions/playground-series-s6e2

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, log_loss
import sklearn.metrics as metrics


# 2. Data Understanding

The dataset contains approximately 630,000 observations with 15 numerical features. The target variable indicates whether a patient has heart disease (1) or not (0).

Before building models, it is important to understand:


*   the distribution of the target variable,
*   relationships between features and the target,

*   potential issues such as missing values or redundant features.

This step helps ensure that the modeling phase is based on meaningful and well-prepared data.


In [ ]:
train = pd.read_csv("/root/.cache/kagglehub/competitions/playground-series-s6e2/train.csv")
train.info()

In [ ]:
train.head(5)

In [ ]:
train.shape

There are 630000 rows in the training dataset, 15 features are all numerical.

# 3. Data Cleaning

In this step:

*   the target column was encoded into a binary variable,
*   unnecessary or redundant columns were removed,
*   the dataset was checked for inconsistencies.

Clean and well-structured data is essential for building stable and interpretable machine-learning models.

In [ ]:
train.describe()

In [ ]:
DiseaseMatch = {"Presence":1, "Absence":0}
train["Disease"] = train["Heart Disease"].map(DiseaseMatch)

In [ ]:
train = train.drop(columns = ["id", "Heart Disease"])

# 4. Exploratory Data Analysis (EDA)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
sns.heatmap(train.corr(), annot = True, annot_kws = {"size":6})

The purpose of EDA is to understand which features are most relevant for predicting heart disease.

Key findings:

1. Maximum Heart Rate (Max HR) has a negative relationship with getting heart disease.

2. Chest Pain type, Exercise angina, ST depression, Slope of ST and Thallium have a positive relationship with getting heart disease.

3. BP, Cholesterol, FBS over 120 show weak connection with getting heart disease.

In [ ]:
sns.countplot(x = "Disease", data = train)

There are around 350000 records for negative heart disease,

290000 records for positive heart disease.

This indicates that the dataset is relatively balanced, which is beneficial for training classification models and reduces the risk of bias toward one class.

# 5. Feature Engineering - Logistic regression


Logistic regression is used as a baseline model because:

1. it is simple and interpretable,

2. it allows us to understand the relationship between features and the probability of heart disease,

3. it provides a good benchmark before trying more complex models.

Regularization was applied to avoid overfitting and improve model generalization.

In [ ]:
X = train.drop(columns = "Disease")
y = train["Disease"]

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state = 42)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns = X_train.columns, index = X_train.index)
X_test = pd.DataFrame(scaler.fit_transform(X_test), columns = X_test.columns, index = X_test.index)

# 6. Model Building - Logistic regression

Different values of the regularization parameter C were tested.

Findings:

* When C ≤ 5.7361e-06, the model is clearly underfitting because the penalty is too strong.

* The best performance is achieved when C ≈ 0.001, where the accuracy stabilizes around 0.882.

This shows that moderate regularization improves model performance while still keeping the model interpretable.






In [ ]:
# Loop through a range of C values to decide which extent of penalty gives the best accuracy
cs = np.logspace(-8,8,30)

for c in cs :
  model = LogisticRegression(
    max_iter=1000,
    random_state = 42,
    solver = "liblinear",
    penalty = "l1", # Using penalty to select the most useful features
    warm_start = True, # keep the coeficients from the previous training
    C = c)
  model.fit(X_train,y_train)
  y_pred = model.predict(X_test)
  accuracy = accuracy_score(y_test, y_pred)
  print(f"The accuracy of C {c} is {accuracy}")

In [ ]:
model.classes_ # {"Presence":1, "Absence":0}

In [ ]:
# Predict the probability of heart disease in the test dataset

test = pd.read_csv("/root/.cache/kagglehub/competitions/playground-series-s6e2/test.csv").drop(columns = "id")
submission = pd.read_csv("/root/.cache/kagglehub/competitions/playground-series-s6e2/sample_submission.csv")

In [ ]:
test = pd.DataFrame(scaler.fit_transform(test), columns = test.columns, index = test.index)

model = LogisticRegression(
    max_iter=1000,
    random_state = 42,
    solver = "liblinear",
    penalty = "l1", # Using penalty to select the most useful features
    warm_start = True, # keep the coeficients from the previous training
    C = 0.001)
model.fit(X_train,y_train)
test_prob = model.predict_proba(test)

In [ ]:
Probs = []
for i in range(len(test_prob)):
  Probs.append(round(test_prob[i][1],3))

In [ ]:
submission["Heart Disease"] = Probs

In [ ]:
submission = submission.set_index("id")

In [ ]:
submission.to_csv("Submission_heartDisease.csv")

# 7. Model Evaluation - Logistic regression

The logistic regression model was evaluated using:

* Accuracy

* ROC curve

* Classification report

* Log loss

The ROC curve shows that the model has good discriminative power, meaning it can distinguish between patients with and without heart disease relatively well.

Plot the ROC curve for logistic regression model.

In [ ]:
from sklearn.metrics import roc_curve, auc

y_prob = model.predict_proba(X_test)[:,1]
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
print(f"AUC: {roc_auc:.4f}")

In [ ]:
import matplotlib.pyplot as plt
plt.figure()
plt.plot(fpr,tpr, label = f"AUC = {roc_auc:.3f}")
plt.plot([0,1], [0,1], linestyle = "--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()

plt.show()

Classification report

In [ ]:
from sklearn.metrics import classification_report
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))


Calculate log loss

In [ ]:
loss = log_loss(y_test, y_prob)
print(f"Log Loss: {loss}")

#8. Model Building - XGBoost

To improve performance, a more advanced model (XGBoost) was implemented.

XGBoost is a powerful gradient-boosting algorithm that is widely used in real-world machine-learning applications because:

* it handles large datasets efficiently,

* it captures non-linear relationships,

* it usually performs better than simple models such as logistic regression.

In [ ]:
import xgboost as xgb

dtrain_clf = xgb.DMatrix(X_train, y_train, enable_categorical = True)
dtest_clf = xgb.DMatrix(X_test, y_test, enable_categorical = True)

params = {"objective": "binary:logistic", "tree_method": "hist"}
n = 1000

evals = [(dtrain_clf, "train"), (dtest_clf, "validation")]

model = xgb.train(
    params = params,
    dtrain = dtrain_clf,
    num_boost_round = n,
    early_stopping_rounds=50,
    evals = evals,
    verbose_eval = 20
)


#9. Model Evaluation - XGBoost

The XGBoost model was evaluated using the same metrics as the logistic regression model to ensure a fair comparison.

Cross-validation was also performed to check whether the model generalizes well to unseen data.

In [ ]:
preds = model.predict(dtest_clf)
loss = log_loss(y_test, preds)
print(f"Log loss: {loss}")


In [ ]:
results = xgb.cv(
   params,
   dtrain_clf,
   num_boost_round=100,
   nfold=3,
   early_stopping_rounds=20,
   metrics = ["auc", "logloss"]
)

In [ ]:
results.head()

# 8. Conclusion

In this project, I built and evaluated multiple machine-learning models to predict the risk of heart disease using a large structured dataset with more than 600,000 observations.

The main results are:

* The logistic regression model achieved a log loss of 0.283, showing strong and stable performance.

* The XGBoost model achieved a log loss of 0.329, which indicates that the simpler model performed better in this case.

* This suggests that the relationships in the dataset are relatively structured and can already be captured well by an interpretable linear model.

More importantly, this project demonstrates the full data-science workflow, including:

* data cleaning,

* exploratory data analysis,

* feature engineering,

* model selection,

* hyperparameter tuning,

*  model evaluation.

From a practical perspective, an interpretable model such as logistic regression can be especially valuable in the healthcare domain because medical professionals need to understand why a prediction is made, not only the prediction itself.

# 11. Further improvement

Future Improvements

To further improve this project, the following steps could be explored:

* Feature selection based on feature importance or statistical tests

* Using cross-validation more extensively for hyperparameter tuning

* Testing additional models such as Random Forest or LightGBM

* Evaluating the model using additional metrics such as recall for high-risk patients